In [1]:
from pathlib import Path
import re

import pandas as pd
from pandas.api.types import CategoricalDtype
# from word2number import w2n # why cant this module be found?

In [2]:
# 1. function: provide dataframe of all products with their attributes parsed from file names
# Data folder paths (constants)
DATA_PATH = Path.cwd().parent.parent / "data" / "cdc" / "raw"
# fetch all file names and connect them to stations
file_paths = list(DATA_PATH.glob("**/*.txt"))
# metadata are ecluded because they are json files
file_names = [fp.name for fp in file_paths]
file_amount = len(file_names)
print(f"Found {file_amount} data files.")

# split file name on every "_" and every "." to gain all intel from the name
product_list = [re.split(r"[_.]", fn) for fn in file_names]
# path/path/example produkt_zehn_min_tu_19910417_19991231_04466.txt

# add file_path as last element to each product entry
for i in range(file_amount):
    product_list[i].append(file_paths[i])
# all appended file paths should be path objects

product_kwargs = {
    "columns": [
        "type",
        "resolution_value",
        "resolution_unit",
        "measurand",
        "from_date",
        "to_date",
        "stations_id",
        "format",
        "path",
    ],
}
product_df = pd.DataFrame(product_list, **product_kwargs)
# parse dates as dates-dtype, resolution value integer and id as integer
product_df["from_date"] = pd.to_datetime(product_df["from_date"], format="%Y%m%d")
product_df["to_date"] = pd.to_datetime(product_df["to_date"], format="%Y%m%d")
product_df["stations_id"] = product_df["stations_id"].astype(int)
# TODO parse resolution_value as integer with package "word2number" because the values are written as words in german

# translate measurand to full name (categorical) sd --> solar radiation,...
measurand_type = CategoricalDtype(categories=["sd", "tu", "ff"])
measurand_name_type = CategoricalDtype(
    categories=["solar radiation", "air temperature", "wind speed"],
)
# translate measurand codes to full names
measurand_mapping = {
    "sd": "solar radiation",
    "tu": "air temperature",
    "ff": "wind speed",
}
product_df["measurand_names"] = (
    product_df["measurand"].map(measurand_mapping).astype(measurand_name_type)
)
product_df["measurand"] = product_df["measurand"].astype(measurand_type)
# all other are strings and can be stored as categorical too
product_df["type"] = product_df["type"].astype("category")
product_df["resolution_unit"] = product_df["resolution_unit"].astype("category")
product_df["format"] = product_df["format"].astype("category")
product_df.head()

Found 209 data files.


,type,resolution_value,resolution_unit,measurand,from_date,to_date,stations_id,format,path,measurand_names
0,produkt,zehn,min,sd,2003-05-13,2009-12-31,4039,txt,/Users/jonas/simultanouesness-analysis/simulta...,solar radiation
1,produkt,zehn,min,sd,2010-01-01,2019-12-31,2115,txt,/Users/jonas/simultanouesness-analysis/simulta...,solar radiation
2,produkt,zehn,min,sd,1995-12-01,1999-12-31,3032,txt,/Users/jonas/simultanouesness-analysis/simulta...,solar radiation
3,produkt,zehn,min,sd,2020-01-01,2024-12-31,6105,txt,/Users/jonas/simultanouesness-analysis/simulta...,solar radiation
4,produkt,zehn,min,sd,2020-01-01,2024-09-30,1200,txt,/Users/jonas/simultanouesness-analysis/simulta...,solar radiation


In [3]:
# 2. join metadata
# Metadata file paths (constants)
META_DATA_TEMPERATURE = DATA_PATH / "station_metadata_temperature.json"
META_DATA_SOLAR = DATA_PATH / "station_metadata_solar.json"
META_DATA_WIND = DATA_PATH / "station_metadata_wind.json"

# station metadata (attributes that are independent from measurements)
# Remove date range columns as they depend on the measurements
# and remove abgabe as it is not relevant for this use case
metadata_temperature = pd.read_json(META_DATA_TEMPERATURE)
metadata_temperature.drop(columns=["von_datum", "bis_datum", "Abgabe"], inplace=True)
print(len(metadata_temperature))
metadata_temperature.set_index(["Stations_id"], inplace=True)

metadata_solar = pd.read_json(META_DATA_SOLAR)
metadata_solar.drop(columns=["von_datum", "bis_datum", "Abgabe"], inplace=True)
metadata_solar.set_index(["Stations_id"], inplace=True)
print(len(metadata_solar))

metadata_wind = pd.read_json(META_DATA_WIND)
metadata_wind.drop(columns=["von_datum", "bis_datum", "Abgabe"], inplace=True)
metadata_wind.set_index(["Stations_id"], inplace=True)
print(len(metadata_wind))

# TODO ensure that these attributes are the same for all metadata files
metadata_station = metadata_temperature.copy()
metadata_station.combine_first(metadata_solar)
metadata_station.combine_first(metadata_wind)
metadata_station

25
17
27


,Stationshoehe,geoBreite,geoLaenge,Stationsname,Bundesland
Stations_id,,,,,
1200,3,54.0691,9.0105,Elpersbüttel,Schleswig-Holstein
1266,18,54.2992,9.3162,Erfde,Schleswig-Holstein
1736,26,53.5731,10.6797,Grambek,Schleswig-Holstein
2115,4,54.1750,7.8920,Helgoland,Schleswig-Holstein
2306,8,54.3194,10.6732,Hohwacht,Schleswig-Holstein
2429,20,53.9897,9.5697,Itzehoe,Schleswig-Holstein
2564,28,54.3776,10.1424,Kiel-Holtenau,Schleswig-Holstein
2907,7,54.7903,8.9514,Leck,Schleswig-Holstein
2961,0,54.4996,10.2737,Leuchtturm Kiel,Schleswig-Holstein


In [4]:
# merge product dataframe with metadata dataframe on station id
product_metadata_df = product_df.merge(
    metadata_station,
    left_on="stations_id",
    right_index=True,
    how="left",
)

# rename columns to english
product_metadata_df.rename(
    columns={
        "Bundesland": "federal_state",
        "Stationshoehe": "altitude",
        "geoBreite": "latitude",
        "geoLaenge": "longitude",
        "Stationsname": "station_name",
    },
    inplace=True,
)

product_metadata_df.set_index(["path"], inplace=True)
product_metadata_df

,type,resolution_value,resolution_unit,measurand,from_date,to_date,stations_id,format,measurand_names,altitude,latitude,longitude,station_name,federal_state
path,,,,,,,,,,,,,,
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/solar/produkt_zehn_min_sd_20030513_20091231_04039.txt,produkt,zehn,min,sd,2003-05-13,2009-12-31,4039,txt,solar radiation,11.0,53.7331,9.8776,Quickborn,Schleswig-Holstein
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/solar/produkt_zehn_min_sd_20100101_20191231_02115.txt,produkt,zehn,min,sd,2010-01-01,2019-12-31,2115,txt,solar radiation,4.0,54.1750,7.8920,Helgoland,Schleswig-Holstein
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/solar/produkt_zehn_min_sd_19951201_19991231_03032.txt,produkt,zehn,min,sd,1995-12-01,1999-12-31,3032,txt,solar radiation,25.0,55.0110,8.4125,List auf Sylt,Schleswig-Holstein
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/solar/produkt_zehn_min_sd_20200101_20241231_06105.txt,produkt,zehn,min,sd,2020-01-01,2024-12-31,6105,txt,solar radiation,15.0,54.3194,9.8051,Ostenfeld (Rendsburg),Schleswig-Holstein
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/solar/produkt_zehn_min_sd_20200101_20240930_01200.txt,produkt,zehn,min,sd,2020-01-01,2024-09-30,1200,txt,solar radiation,3.0,54.0691,9.0105,Elpersbüttel,Schleswig-Holstein
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/air_temperature/produkt_zehn_min_tu_20100101_20191231_01266.txt,produkt,zehn,min,tu,2010-01-01,2019-12-31,1266,txt,air temperature,18.0,54.2992,9.3162,Erfde,Schleswig-Holstein
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/air_temperature/produkt_zehn_min_tu_20050706_20091231_02306.txt,produkt,zehn,min,tu,2005-07-06,2009-12-31,2306,txt,air temperature,8.0,54.3194,10.6732,Hohwacht,Schleswig-Holstein
/Users/jonas/simultanouesness-analysis/simultaneousness-analysis/data/cdc/raw/air_temperature/produkt_zehn_min_tu_20200101_20241231_05930.txt,produkt,zehn,min,tu,2020-01-01,2024-12-31,5930,txt,air temperature,1.0,54.6410,10.0238,Schönhagen (Ostseebad),Schleswig-Holstein


In [5]:
# 3. function: "simple" filter dataframe by measurand, date range, station id and return list of paths
measurand_names_filter_parameter = []
measurand_names_filter_var = (
    measurand_names_filter_parameter
    if measurand_names_filter_parameter != []
    else product_metadata_df["measurand_names"]
)

from_date_filter_parameter = pd.to_datetime("2012-01-01")
from_date_filter_var = (
    from_date_filter_parameter
    if from_date_filter_parameter is not None
    else product_metadata_df["from_date"]
)

to_date_filter_parameter = pd.to_datetime("2022-12-31")
to_date_filter_var = (
    to_date_filter_parameter
    if to_date_filter_parameter is not None
    else product_metadata_df["to_date"]
)

station_id_filter_parameter = [1200]
station_id_filter_var = (
    station_id_filter_parameter
    if station_id_filter_parameter != []
    else product_metadata_df["stations_id"]
)

station_name_filter_parameter = [1200]
station_name_filter_var = (
    station_name_filter_parameter
    if station_name_filter_parameter != []
    else product_metadata_df["station_name"]
)

federal_state_filter_parameter = [1200]
federal_state_filter_var = (
    federal_state_filter_parameter
    if federal_state_filter_parameter != []
    else product_metadata_df["federal_state"]
)

# possible filter parameters:
# measurand ("code")
# resolution (value + unit)
# format
# type
# Area (altitude, latitude, longitude and PLZ,.. after merging with additional metadata)

filtered = product_metadata_df[
    (product_metadata_df["measurand_names"].isin(measurand_names_filter_var))
    & (
        product_metadata_df["to_date"] >= from_date_filter_var
    )  # crossover of from and to is correct !
    & (
        product_metadata_df["from_date"] <= to_date_filter_var
    )  # crossover of from and to is correct !
    & (product_metadata_df["stations_id"].isin(station_id_filter_var))
]
testlist = filtered.index.tolist()
testlist

for station in stations:
    for measureand in measurands:
        for range in ranges:
            read data for station x and measurand y and range z
            store in list
        concat data of all ranges for measurand y and station x
    merge measurand on timestemp and station
concat all stations
cut to filtered range

# per station and than concat

# next function is to conact data to a single dataframe
result_df = pd.DataFrame()
imported_list = []
import_index = 0
for path in testlist:
    temp_df = pd.read_csv(
        path,
        sep=";",
        parse_dates=["MESS_DATUM"],
        date_format="%Y%m%d%H%M",
        index_col=["MESS_DATUM", "STATIONS_ID"],
    )
    # rename QN to QN_measurand to make it unique
    temp_df.drop(columns=["  QN", "eor"], inplace=True)
    if import_index == 0:
        result_df = temp_df
        import_index += 1
        continue
    result_df = result_df.merge(temp_df, on=["MESS_DATUM", "STATIONS_ID"], how="outer")
    import_index += 1
# drop all index where MESS_DATUM is not in date range

result_df = result_df[
    (result_df.index.get_level_values("MESS_DATUM") >= from_date_filter_var)
    & (result_df.index.get_level_values("MESS_DATUM") <= to_date_filter_var)
]
result_df = result_df.unstack(level=["STATIONS_ID"])
result_df
# read csv
# rename QN to QN_FF oder SD,...
# join together by datetime index (outer join?)

# are different presets nessasary for different measurands?
# concat per measurand
# rename QN to QN_FF oder SD,...
# if there ist more than measurand and than merge together by datetime index

# parameter to return possible arguments for measurand, date range


# return mathed dateframe request 2088-2022 --> 2003 - 2025 (if in stock more, if not in stock less)
# return station ids
# cache dataframe (2pikle, pakee, dether, etc.) look IO Tools, pandas IO
# data class (dtype) like api
# kwags or arguments as paramete

SyntaxError: invalid syntax (300941743.py, line 67)

In [ ]:
DF1 = pd.DataFrame(
    {
        "Timestemp": [1, 2, 3, 4, 5, 6],
        "Station": ["A", "A", "A", "A", "A", "A"],
        "Value1": range(6),
    },
)
DF2 = pd.DataFrame(
    {
        "Timestemp": [7, 8, 9, 10, 11, 12],
        "Station": ["A", "A", "A", "A", "A", "A"],
        "Value1": range(6),
    },
)
DF3 = pd.DataFrame(
    {
        "Timestemp": [7, 8, 9, 10, 11, 12],
        "Station": ["A", "A", "A", "A", "A", "A"],
        "Value2": range(6),
    },
)
DF2 = pd.concat(
    [DF1, DF2], ignore_index=True
)  # concat all timestemps for every measureand per station
DF3 = DF2.merge(
    DF3, on=["Timestemp", "Station"], how="outer"
)  # merge all measurand of a station
# concant all stations together
# unstack by stations
DF3

,Timestemp,Station,Value1,Value2
0,1,A,0,NaN
1,2,A,1,NaN
2,3,A,2,NaN
3,4,A,3,NaN
4,5,A,4,NaN
5,6,A,5,NaN
6,7,A,0,0.0
7,8,A,1,1.0
8,9,A,2,2.0
9,10,A,3,3.0
